# Converting Biomedical Datasets to SpERT Format
This notebook converts the original ADKG and MDKG biomedical datasets into the JSON format required by the SpERT relation extraction framework.

In [94]:
import torch
import transformers

print(torch.__version__)
print(transformers.__version__)

2.8.0+cu128
4.57.6


## Environment and Dependency Verification
This section verifies the installed PyTorch and Transformers versions used for the biomedical NLP pipeline.

In [95]:
import json
from transformers import AutoTokenizer
from tqdm import tqdm

## Library Imports
The required preprocessing libraries and BioBERT tokenizer utilities are imported.

In [96]:
tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1"
)

## Initialize BioBERT Tokenizer
The BioBERT tokenizer is loaded to convert biomedical text into transformer-compatible token representations.

In [97]:
with open("../data/ADKG.json") as f:
    adkg = json.load(f)

with open("../data/MDKG.json") as f:
    mdkg = json.load(f)

## Load Raw Biomedical Datasets
The original ADKG and MDKG datasets are loaded from disk.

In [98]:
def tokenize_with_offsets(text):
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=False
    )

    return encoding.tokens(), encoding["offset_mapping"]

## Tokenization with Offset Mapping
This helper function tokenizes biomedical text while preserving character-level offsets needed for entity span alignment.

In [99]:
sample = adkg["train"][0]

tokens, offsets = tokenize_with_offsets(sample["text"])

for t, o in zip(tokens[:30], offsets[:30]):
    print(f"{t:15} {o}")

the             (0, 3)
inter           (4, 9)
##lock          (9, 13)
##ing           (13, 16)
finger          (17, 23)
test            (24, 28)
in              (29, 31)
patients        (32, 40)
with            (41, 45)
park            (46, 50)
##ins           (50, 53)
##on            (53, 55)
'               (55, 56)
s               (56, 57)
disease         (58, 65)
and             (66, 69)
healthy         (70, 77)
subjects        (78, 86)
.               (86, 87)


## Entity Span Conversion
The following code maps raw entity annotations to token-level spans required by SpERT.

In [100]:
def char_to_token_span(start_char, end_char, offsets):

    token_start = None
    token_end = None

    for idx, (start, end) in enumerate(offsets):

        if start <= start_char < end:
            token_start = idx

        if start < end_char <= end:
            token_end = idx

    return token_start, token_end

## Relation Conversion
Relation annotations are transformed into the structured format expected by the SpERT framework.

In [101]:
entity = sample["entities"][1]

print(entity)

token_start, token_end = char_to_token_span(
    entity["start"],
    entity["end"],
    offsets
)

print("\nToken span:")
print(token_start, token_end)

print("\nRecovered tokens:")
print(tokens[token_start:token_end+1])

{'id': 'T1', 'type': 'disease', 'start': 46, 'end': 65, 'text': "Parkinson's disease"}

Token span:
9 14

Recovered tokens:
['park', '##ins', '##on', "'", 's', 'disease']


## Sample Conversion Function
This function converts an individual biomedical sentence into the final SpERT-compatible format.

In [102]:
def convert_entities(sample, offsets):

    entities = []
    entity_id_map = {}

    for idx, ent in enumerate(sample["entities"]):

        token_start, token_end = char_to_token_span(
            ent["start"],
            ent["end"],
            offsets
        )

        # Skip entities outside truncated region
        if token_start is None or token_end is None:
            continue

        converted = {
            "type": ent["type"],
            "start": token_start,
            "end": token_end + 1
        }

        entity_id_map[ent["id"]] = len(entities)

        entities.append(converted)

    return entities, entity_id_map

## Dataset Conversion Pipeline
The entire dataset is processed split-by-split to generate train, development, and test files.

In [103]:
entities, entity_id_map = convert_entities(sample, offsets)

print(entities[:5])
print(entity_id_map)

[{'type': 'method', 'start': 1, 'end': 6}, {'type': 'disease', 'start': 9, 'end': 15}]
{'T18': 0, 'T1': 1}


## Save Converted Datasets
Converted datasets are saved to disk for downstream training and evaluation.

In [104]:
def convert_relations(sample, entity_id_map):

    relations = []

    for rel in sample["relations"]:

        head_id = rel["head"]["id"]
        tail_id = rel["tail"]["id"]

        # Skip truncated entities
        if head_id not in entity_id_map:
            continue

        if tail_id not in entity_id_map:
            continue

        converted = {
            "type": rel["type"],
            "head": entity_id_map[head_id],
            "tail": entity_id_map[tail_id]
        }

        relations.append(converted)

    return relations

## Generate Entity and Relation Type Definitions
This section extracts ontology metadata including entity and relation types.

In [105]:
relations = convert_relations(sample, entity_id_map)

print(relations)

[{'type': 'help_diagnose', 'head': 0, 'tail': 1}]


## Sequence Length Verification
The final preprocessing step verifies that token lengths remain within BioBERT's maximum context size.

In [106]:
def convert_sample(sample):

    text = sample["text"]

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=512,
        padding=False
    )

    tokens = tokenizer.convert_ids_to_tokens(
        encoding["input_ids"]
    )

    offsets = encoding["offset_mapping"]
    
    # hard safety cutoff
    tokens = tokens[:512]
    offsets = offsets[:512]



    entities, entity_id_map = convert_entities(
        sample,
        offsets
    )

    relations = convert_relations(
        sample,
        entity_id_map
    )

    return {
        "tokens": tokens,
        "entities": entities,
        "relations": relations
    }

## Final Preprocessing Summary
The outputs below confirm successful dataset conversion and preprocessing completion.

In [107]:
converted = convert_sample(sample)

print(converted.keys())

print("\nTOKENS:")
print(converted["tokens"][:20])

print("\nENTITIES:")
print(converted["entities"])

print("\nRELATIONS:")
print(converted["relations"])

dict_keys(['tokens', 'entities', 'relations'])

TOKENS:
['[CLS]', 'the', 'inter', '##lock', '##ing', 'finger', 'test', 'in', 'patients', 'with', 'park', '##ins', '##on', "'", 's', 'disease', 'and', 'healthy', 'subjects', '.']

ENTITIES:
[{'type': 'method', 'start': 2, 'end': 7}, {'type': 'disease', 'start': 10, 'end': 16}]

RELATIONS:
[{'type': 'help_diagnose', 'head': 0, 'tail': 1}]


In [108]:
def convert_dataset(dataset):

    converted_data = {}

    for split in ["train", "dev", "test"]:

        converted_split = []

        for sample in tqdm(dataset[split]):

            try:
                converted = convert_sample(sample)
                converted_split.append(converted)

            except Exception as e:

                print("Error processing sample:")
                print(sample["text"])
                print(e)

        converted_data[split] = converted_split

    return converted_data

In [109]:
adkg_spert = convert_dataset(adkg)

100%|██████████| 1220/1220 [00:00<00:00, 5904.51it/s]


In [110]:
mdkg_spert = convert_dataset(mdkg)

100%|██████████| 912/912 [00:00<00:00, 5724.11it/s]


In [111]:
for split in ["train", "dev", "test"]:

    print(split)

    print("ADKG:", len(adkg_spert[split]))
    print("MDKG:", len(mdkg_spert[split]))

train
ADKG: 5605
MDKG: 4825
dev
ADKG: 1206
MDKG: 941
test
ADKG: 1220
MDKG: 912


In [112]:
mkdir -p ../data/spert_format

In [113]:
with open("../data/spert_format/ADKG_spert.json", "w") as f:
    json.dump(adkg_spert, f, indent=2)

In [114]:
with open("../data/spert_format/MDKG_spert.json", "w") as f:
    json.dump(mdkg_spert, f, indent=2)

In [115]:
import os

os.makedirs("../data/spert_adkg", exist_ok=True)
os.makedirs("../data/spert_mdkg", exist_ok=True)

In [116]:
for split in ["train", "dev", "test"]:

    with open(f"../data/spert_adkg/{split}.json", "w") as f:
        json.dump(adkg_spert[split], f, indent=2)

In [117]:
for split in ["train", "dev", "test"]:

    with open(f"../data/spert_mdkg/{split}.json", "w") as f:
        json.dump(mdkg_spert[split], f, indent=2)

In [118]:
def extract_types(dataset):

    entity_types = set()
    relation_types = set()

    for split in dataset:

        for sample in dataset[split]:

            for ent in sample["entities"]:
                entity_types.add(ent["type"])

            for rel in sample["relations"]:
                relation_types.add(rel["type"])

    return sorted(entity_types), sorted(relation_types)

In [119]:
adkg_entity_types, adkg_relation_types = extract_types(adkg_spert)

print(adkg_entity_types)
print(adkg_relation_types)

['disease', 'drug', 'gene', 'method', 'mutation', 'other']
['abbreviation_for', 'associated_with', 'characteristic_of', 'help_diagnose', 'hyponym_of', 'risk_factor_of', 'treatment_for', 'treatment_target_for']


In [120]:
def create_types_json(entity_types, relation_types):

    types = {
        "entities": {},
        "relations": {}
    }

    for ent in entity_types:
        types["entities"][ent] = {
            "short": ent,
            "verbose": ent
        }

    for rel in relation_types:
        types["relations"][rel] = {
            "short": rel,
            "verbose": rel,
            "symmetric": False
        }

    return types

In [121]:
adkg_types = create_types_json(
    adkg_entity_types,
    adkg_relation_types
)

with open("../data/spert_adkg/types.json", "w") as f:
    json.dump(adkg_types, f, indent=2)

In [122]:
mdkg_entity_types, mdkg_relation_types = extract_types(mdkg_spert)

mdkg_types = create_types_json(
    mdkg_entity_types,
    mdkg_relation_types
)

with open("../data/spert_mdkg/types.json", "w") as f:
    json.dump(mdkg_types, f, indent=2)

In [123]:
max_len = 0

for split in ["train", "dev", "test"]:
    for sample in adkg_spert[split]:
        l = len(sample["tokens"])
        max_len = max(max_len, l)

print("Max token length:", max_len)

Max token length: 292


In [124]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-PCIE-40GB MIG 2g.10gb
